# Module 05 — Kiểm định t (độc lập & bắt cặp)
Notebook RIÊNG cho Module 05 — tái lập cả 3 ví dụ t-test trong bài: độc lập có ý nghĩa (phương pháp dạy), độc lập không có ý nghĩa (giới tính), và bắt cặp (pretest/posttest).

Tương ứng SPSS/PSPP: `Analyze > Compare Means > Independent-Samples T Test` / `Paired-Samples T Test`.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

In [ ]:
df["gain"] = df["posttest"] - df["pretest"]

## 1. T-test độc lập: phương pháp dạy → mức tăng điểm (`gain`) — CÓ ý nghĩa
Bước 1 luôn là Levene's Test: quyết định đọc dòng "Equal variances assumed" hay "not assumed".

🎯 **Phương pháp này trả lời câu hỏi gì?** T-test độc lập kiểm tra: sự khác biệt trung bình QUAN SÁT ĐƯỢC giữa 2 nhóm KHÔNG liên quan (không phải cùng người) có đủ lớn để tin là khác biệt THẬT trong tổng thể hay không, hay chỉ là biến động ngẫu nhiên do lấy mẫu. Giả thuyết không (H0): 2 nhóm có cùng trung bình tổng thể.

In [ ]:
g1 = df.loc[df["method"]=="Dự án", "gain"]
g2 = df.loc[df["method"]=="Truyền thống", "gain"]
lev_stat, lev_p = stats.levene(g1, g2)
t, p = stats.ttest_ind(g1, g2, equal_var=(lev_p >= 0.05))
print(f"Levene: F={lev_stat:.3f}, p={lev_p:.3f} -> {'Equal variances assumed' if lev_p>=0.05 else 'not assumed'}")
print(f"Dự án: n={len(g1)}, M={g1.mean():.2f}, SD={g1.std(ddof=1):.2f}")
print(f"Truyền thống: n={len(g2)}, M={g2.mean():.2f}, SD={g2.std(ddof=1):.2f}")
print(f"t({len(g1)+len(g2)-2})={t:.3f}, p={p:.2e}")

#### 📤 Đầu ra thật
Levene p=0,535 (≥0,05 → đọc dòng "assumed"). t(238)=7,589, p<0,001. Dự án M=8,66 SD=5,77 (n=115); Truyền thống M=2,89 SD=6,00 (n=125). ✅ Khớp đúng bài.

**Viết chuẩn APA:** "Mức tăng điểm trung bình của nhóm Dự án (M=8,66, SD=5,77) cao hơn có ý nghĩa thống kê (t(238)=7,59, p<0,001) so với nhóm Truyền thống (M=2,89, SD=6,00)."

## 2. T-test độc lập: giới tính → điểm Toán — KHÔNG có ý nghĩa

In [ ]:
mA = df.loc[df["gender"]=="Nam", "math"]
mB = df.loc[df["gender"]=="Nữ", "math"]
lev_stat2, lev_p2 = stats.levene(mA, mB)
t2, p2 = stats.ttest_ind(mA, mB, equal_var=(lev_p2 >= 0.05))
print(f"Levene: F={lev_stat2:.3f}, p={lev_p2:.3f}")
print(f"Nam: n={len(mA)}, M={mA.mean():.2f}, SD={mA.std(ddof=1):.2f}")
print(f"Nữ: n={len(mB)}, M={mB.mean():.2f}, SD={mB.std(ddof=1):.2f}")
print(f"t({len(mA)+len(mB)-2})={t2:.3f}, p={p2:.3f}")

#### 📤 Đầu ra thật
Levene p=0,081 (≥0,05 nhưng khá gần ngưỡng — nên thận trọng). t(238)=-1,348, p=0,179 → **không đủ bằng chứng** về khác biệt giới tính trong điểm Toán ở mẫu này. Một kết quả "không tìm thấy" cũng cần báo cáo trung thực, không "đào bới" (p-hack) tới khi có ý nghĩa.

## 3. T-test bắt cặp: pretest → posttest (cùng 240 học sinh)
Bắt cặp mạnh hơn độc lập vì loại bỏ được biến thiên CÁ NHÂN giữa các học sinh — chỉ còn lại biến thiên do tác động của việc học.

🎯 **Phương pháp này trả lời câu hỏi gì?** T-test bắt cặp kiểm tra: khi CÙNG một nhóm đối tượng được đo 2 lần (trước/sau, hoặc 2 biến khác nhau), độ CHÊNH LỆCH trung bình giữa 2 lần đo có đủ lớn để tin là một thay đổi THẬT (không phải do may rủi) hay không. H0: chênh lệch trung bình = 0.

In [ ]:
t3, p3 = stats.ttest_rel(df["posttest"], df["pretest"])
diff = df["posttest"] - df["pretest"]
print(f"Mean pretest={df['pretest'].mean():.2f} (SD={df['pretest'].std(ddof=1):.2f})")
print(f"Mean posttest={df['posttest'].mean():.2f} (SD={df['posttest'].std(ddof=1):.2f})")
print(f"Chênh lệch trung bình={diff.mean():.2f}")
print(f"t({len(df)-1})={t3:.2f}, p={p3:.2e}")
r_pp = stats.pearsonr(df["pretest"], df["posttest"])[0]
print(f"Tương quan pretest-posttest r={r_pp:.3f} (giải thích vì sao bắt cặp mạnh hơn độc lập)")

#### 📤 Đầu ra thật
Chênh lệch trung bình=5,65, t(239)=13,37, p<0,001. Tương quan pretest-posttest r≈0,857 — rất cao, đúng lý do bắt cặp mạnh hơn độc lập rất nhiều với cùng cỡ mẫu. ✅ Khớp đúng bài.